# Teacher B — SeamlessM4T v2

## Copy/run order

1. Block 1 — configuration
2. Block 2 — Phase 0A runtime probe
3. Block 3 — Phase 0B bundle and audio verification
4. Block 4 — stop gate
5. Block 5 — dependencies (after approval)
6. Block 6 — shared utilities
7. Block 7 — immutable model revision
8. Block 8 — SeamlessM4T generation
9. Block 9 — tokenizer-risk report
10. Block 10 — metadata

This notebook independently benchmarks English speech → Hindi text using `facebook/seamless-m4t-v2-large`. It does not generate Hindi speech, use Nadantara's vocoder, or modify any source manifest.

Run the Phase 0 runtime/file cells first and stop for approval. The model cell is deliberately later. Official API: [SeamlessM4T v2](https://huggingface.co/facebook/seamless-m4t-v2-large) and [Transformers documentation](https://huggingface.co/docs/transformers/main/en/model_doc/seamless_m4t_v2).

In [ ]:
# BLOCK 1 — configuration and benchmark bundle paths.
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import statistics
import tarfile
import time
import unicodedata

import numpy as np
import torch

MODEL_ID = 'facebook/seamless-m4t-v2-large'
BUNDLE_ARCHIVE = Path('/content/nadantara-benchmark-smoke.tar.gz')
BUNDLE_ROOT = Path('/content/nadantara-benchmark-smoke')
OUTPUT_ROOT = Path('/content/teacher_b_outputs')
RETRY_FAILED = False
MAX_AUDIO_SECONDS = 30.0
MAX_TRANSLATION_TOKENS = 256
print('python=', platform.python_version())
print('model=', MODEL_ID)

In [ ]:
# BLOCK 2 — PHASE 0A runtime probe; this cell downloads no models.
import shutil
print('python=', platform.python_version())
print('torch=', torch.__version__)
print('torch_cuda_version=', torch.version.cuda)
print('cuda_available=', torch.cuda.is_available())
print('content_exists=', Path('/content').is_dir())
print('free_disk_gib=', round(shutil.disk_usage('/content').free / 1024**3, 3))
if not torch.cuda.is_available():
    raise RuntimeError('PHASE 0 STOP: no CUDA GPU is available')
device_index = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device_index)
free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
print('gpu_index=', device_index)
print('gpu_name=', props.name)
print('gpu_vram_gib=', round(props.total_memory / 1024**3, 3))
print('gpu_free_memory_gib=', round(free_bytes / 1024**3, 3))
print('gpu_total_memory_gib=', round(total_bytes / 1024**3, 3))
print('PHASE_0_RUNTIME_PASS')

In [ ]:
# BLOCK 3 — PHASE 0B uploaded-file access, manifest identity, and audio-byte checks.
if not BUNDLE_ARCHIVE.is_file():
    raise FileNotFoundError(f'upload the benchmark archive first: {BUNDLE_ARCHIVE}')
def safe_extract(archive: Path, destination: Path) -> None:
    with tarfile.open(archive, 'r:gz') as tar:
        root = destination.parent.resolve()
        for member in tar.getmembers():
            target = (destination.parent / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        tar.extractall(destination.parent)
if not (BUNDLE_ROOT / 'benchmark_manifest.jsonl').is_file():
    safe_extract(BUNDLE_ARCHIVE, BUNDLE_ROOT)
manifest_path = BUNDLE_ROOT / 'benchmark_manifest.jsonl'
rows = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
if not rows or len({row['sample_id'] for row in rows}) != len(rows):
    raise RuntimeError('empty or duplicate benchmark manifest')
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1024 * 1024): digest.update(chunk)
    return digest.hexdigest()
for row in rows:
    audio = BUNDLE_ROOT / row['audio_path']
    if not audio.is_file(): raise FileNotFoundError(audio)
    if sha256_file(audio) != row['audio_sha256']:
        raise RuntimeError(f'audio hash mismatch for {row["sample_id"]}')
print('samples=', len(rows))
print('sample_ids=', [row['sample_id'] for row in rows])
print('bundle_metadata=', json.loads((BUNDLE_ROOT / 'bundle_metadata.json').read_text(encoding='utf-8')))
print('PHASE_0_FILES_PASS')

## BLOCK 4 — Phase 0 stop gate

Confirm `PHASE_0_RUNTIME_PASS` and `PHASE_0_FILES_PASS`, then inspect the actual GPU, free VRAM, selected IDs, and audio hashes. Stop here until the teacher smoke test is approved. The next cell installs only this notebook's dependencies.

In [ ]:
# BLOCK 5 — POST-APPROVAL ONLY — Teacher B's isolated dependencies.
%pip install -q 'transformers==5.16.1' accelerate sentencepiece soundfile scipy psutil huggingface_hub sacrebleu

In [ ]:
# BLOCK 6 — shared Teacher B utilities.
import soundfile as sf
from scipy.signal import resample_poly
TARGET_RATE = 16_000
def normalize_hindi(text: str) -> str:
    return ' '.join(unicodedata.normalize('NFC', text).split())
def read_audio(path: Path) -> tuple[np.ndarray, float, int]:
    array, sample_rate = sf.read(path, dtype='float32', always_2d=True)
    waveform = array.mean(axis=1) if array.shape[1] > 1 else array[:, 0]
    if sample_rate != TARGET_RATE:
        divisor = math.gcd(int(sample_rate), TARGET_RATE)
        waveform = resample_poly(waveform, TARGET_RATE // divisor, int(sample_rate) // divisor).astype('float32')
        sample_rate = TARGET_RATE
    duration = len(waveform) / TARGET_RATE
    if duration > MAX_AUDIO_SECONDS: raise ValueError(f'audio exceeds {MAX_AUDIO_SECONDS}s: {duration:.3f}s')
    return np.clip(waveform, -1.0, 1.0).astype('float32'), duration, int(sample_rate)
def existing_records(path: Path) -> dict[str, dict[str, object]]:
    if not path.is_file(): return {}
    return {str(row['sample_id']): row for row in (json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip())}
def append_jsonl(path: Path, row: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n'); stream.flush()
def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
durations = [read_audio(BUNDLE_ROOT / row['audio_path'])[1] for row in rows]
print('transformers=', __import__('transformers').__version__)
print('duration_min_median_p90_p95_max=', min(durations), statistics.median(durations), np.percentile(durations, 90), np.percentile(durations, 95), max(durations))

In [ ]:
# BLOCK 7 — POST-APPROVAL ONLY — resolve and record the exact model revision.
from huggingface_hub import HfApi
HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get('HF_TOKEN')
except Exception:
    pass
MODEL_REVISION = HfApi(token=HF_TOKEN).model_info(MODEL_ID, token=HF_TOKEN).sha
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'HF_TOKEN_available': bool(HF_TOKEN)})

In [ ]:
# BLOCK 8 — POST-APPROVAL ONLY — batch-1 deterministic speech-to-Hindi-text generation.
from transformers import AutoProcessor, SeamlessM4Tv2Model

output_path = OUTPUT_ROOT / 'teacher_seamless_m4t_v2.jsonl'
records = existing_records(output_path)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
generation_config = {'src_lang': 'eng', 'tgt_lang': 'hin', 'generate_speech': False, 'text_num_beams': 4, 'text_do_sample': False, 'max_new_tokens': MAX_TRANSLATION_TOKENS, 'batch_size': 1}
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN)
model = SeamlessM4Tv2Model.from_pretrained(MODEL_ID, revision=MODEL_REVISION, torch_dtype=dtype, low_cpu_mem_usage=True, device_map='auto', token=HF_TOKEN)
model.eval()
def generated_ids(value):
    sequences = getattr(value, 'sequences', None)
    if sequences is not None: value = sequences
    elif isinstance(value, (tuple, list)): value = value[0]
    if value is None: raise RuntimeError('Seamless generation returned no token sequences')
    if not isinstance(value, torch.Tensor): value = torch.as_tensor(value)
    value = value.detach().cpu()
    if value.ndim == 1: return value.tolist()
    if value.ndim == 2: return value[0].tolist()
    raise RuntimeError(f'unexpected Seamless token shape: {tuple(value.shape)}')
model_device = model.device
model_dtype = next(parameter for parameter in model.parameters() if parameter.is_floating_point()).dtype
for row in rows:
    sample_id = row['sample_id']
    if sample_id in records and (records[sample_id].get('status') == 'ok' or not RETRY_FAILED): continue
    base = {'schema_version': 1, 'sample_id': sample_id, 'audio_path': row['audio_path'], 'audio_sha256': row['audio_sha256'], 'source_manifest_sha256': row.get('source_manifest_sha256'), 'source_dataset': row.get('source_dataset'), 'source_dataset_revision': row.get('source_dataset_revision'), 'source_license': row.get('source_license'), 'source_url': row.get('source_url'), 'source_split': row.get('source_split'), 'reference_hindi': row.get('reference_hindi'), 'source_text': row.get('source_text'), 'source_asr_text': row.get('source_asr_text'), 'teacher_pipeline': MODEL_ID, 'speech_teacher': MODEL_ID, 'speech_teacher_revision': MODEL_REVISION, 'speech_teacher_license': 'CC-BY-NC-4.0', 'translation_teacher': None, 'status': 'error', 'stage': 'seamless', 'generation_config': generation_config}
    started = time.perf_counter()
    try:
        waveform, duration, _ = read_audio(BUNDLE_ROOT / row['audio_path'])
        inputs = processor(audio=waveform, sampling_rate=TARGET_RATE, return_tensors='pt')
        inputs = {key: (value.to(device=model_device, dtype=model_dtype) if torch.is_floating_point(value) else value.to(model_device)) for key, value in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(**inputs, tgt_lang='hin', generate_speech=False, text_num_beams=4, text_do_sample=False, max_new_tokens=MAX_TRANSLATION_TOKENS)
        raw_text = processor.decode(generated_ids(generated), skip_special_tokens=True)
        base.update({'status': 'ok', 'stage': None, 'teacher_hindi_model_decoded_raw': raw_text, 'teacher_hindi_raw': raw_text, 'teacher_hindi_normalized': normalize_hindi(raw_text), 'translation_seconds': time.perf_counter() - started, 'duration_seconds': duration})
    except torch.cuda.OutOfMemoryError as exc:
        clear_cuda(); base.update({'error_type': 'cuda_oom', 'error': repr(exc), 'translation_seconds': time.perf_counter() - started})
    except Exception as exc:
        base.update({'error_type': type(exc).__name__, 'error': repr(exc), 'translation_seconds': time.perf_counter() - started})
    append_jsonl(output_path, base); records[sample_id] = base
    print(sample_id, base['status'], base.get('error_type', 'ok'), base.get('error', ''))
del model, processor
clear_cuda()
print('canonical_output=', output_path)

In [ ]:
# BLOCK 9 — optional pre-distillation tokenizer-risk report. Upload the exact baseline hindi_2k.model first.
TOKENIZER_PATH = Path('/content/hindi_2k.model')
MAX_SEMANTIC_TOKENS = 128
if TOKENIZER_PATH.is_file():
    import sentencepiece as spm
    tokenizer = spm.SentencePieceProcessor(model_file=str(TOKENIZER_PATH))
    outputs = [json.loads(line) for line in output_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    lengths = []; unknown = 0; total = 0
    for row in outputs:
        if row.get('status') != 'ok': continue
        ids = tokenizer.encode(str(row.get('teacher_hindi_raw', '')), out_type=int)
        lengths.append(len(ids)); unknown += sum(token == tokenizer.unk_id() for token in ids); total += len(ids)
    print({'vocab_size': tokenizer.vocab_size(), 'unknown_rate': unknown / max(1, total), 'p50': np.percentile(lengths, 50) if lengths else None, 'p90': np.percentile(lengths, 90) if lengths else None, 'p95': np.percentile(lengths, 95) if lengths else None, 'p99': np.percentile(lengths, 99) if lengths else None, 'max': max(lengths) if lengths else None, 'exceeding_max_semantic_tokens': sum(length + 1 > MAX_SEMANTIC_TOKENS for length in lengths) / max(1, len(lengths))})
else:
    print('tokenizer report skipped: upload exact baseline tokenizer to', TOKENIZER_PATH)

In [ ]:
# BLOCK 10 — persist reproducibility metadata without secrets.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
metadata = {'schema_version': 1, 'teacher_pipeline': MODEL_ID, 'model_id': MODEL_ID, 'model_revision': globals().get('MODEL_REVISION'), 'model_license': 'CC-BY-NC-4.0', 'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': __import__('transformers').__version__, 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu', 'precision': str(dtype), 'bundle_metadata': json.loads((BUNDLE_ROOT / 'bundle_metadata.json').read_text(encoding='utf-8')), 'generation_config': globals().get('generation_config', {}), 'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
(OUTPUT_ROOT / 'experiment_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(metadata, indent=2, ensure_ascii=False, sort_keys=True))
print('Teacher B complete; preserve teacher_seamless_m4t_v2.jsonl and compare only against the same benchmark IDs.')